# Theoretical Analysis: Why Hierarchical Quilting Fails

**DSC 205 — Cluster Quilting (Zheng, Chang & Allen, 2024)**

This notebook provides a rigorous linear algebra treatment of the alignment step in Cluster Quilting, explains why the sequential chain compounds errors, why the hierarchical variant was *supposed* to fix this, and why it introduces a different — and empirically worse — failure mode. We demonstrate each failure with minimal reproducible examples and propose potential solutions.

---

## Table of Contents

1. [Background: The Alignment Problem](#1)
2. [Linear Algebra of Patchwise SVD and Alignment](#2)
3. [Sequential Quilting: Error Propagation Along the Chain](#3)
4. [Hierarchical Quilting: The Promise](#4)
5. [Hierarchical Quilting: The Failure — Greedy Locality → Global Separation](#5)
6. [Demonstration: SVD Alignment Failure Modes](#6)
7. [Demonstration: Chain vs. Tree Error Accumulation](#7)
8. [Demonstration: The Global Separation Problem](#8)
9. [Potential Solutions](#9)
10. [Summary](#10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.linalg import orthogonal_procrustes, svd, lstsq
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'font.size': 9,
})

rng = np.random.default_rng(42)
print('Setup complete.')

<a id='1'></a>
## 1. Background: The Alignment Problem

Cluster Quilting operates on **patchwork data**: a collection of patches $\{X^{(m)}\}_{m=1}^{M}$ where each patch $X^{(m)} \in \mathbb{R}^{n_m \times p_m}$ observes a subset of samples on a subset of features. Patches share **overlapping samples** (rows) but have mostly **disjoint features** (columns).

The goal is to recover a low-rank embedding $\widetilde{U} \in \mathbb{R}^{N \times r}$ of all $N$ samples in a common $r$-dimensional subspace, even though no single patch observes all samples on all features.

### The core difficulty

Each patch's truncated SVD yields left singular vectors $U^{(m)} \in \mathbb{R}^{n_m \times r}$ that span the best rank-$r$ approximation of that patch. But **SVD bases are unique only up to orthogonal transformation**: if $U^{(m)}$ is a valid set of left singular vectors, so is $U^{(m)} Q$ for any orthogonal $Q \in O(r)$. This includes:

- **Sign flips** (reflections): any column of $U$ can be negated
- **Rotations**: the entire basis can be rotated within the $r$-dimensional subspace
- **Permutations**: columns can be reordered (when singular values coincide)

Two patches observing the *same* underlying cluster structure will produce $U$ matrices that are related by an unknown orthogonal transformation $Q$. Alignment recovers this $Q$ using **shared samples** — rows observed in both patches.

<a id='2'></a>
## 2. Linear Algebra of Patchwise SVD and Alignment

### 2.1 Patchwise SVD

For patch $m$, compute the thin SVD:

$$X^{(m)} = U^{(m)} \Sigma^{(m)} V^{(m)\top}$$

Truncate to rank $r$ (typically $r = K$, the number of clusters):

$$X^{(m)} \approx U_r^{(m)} \Sigma_r^{(m)} V_r^{(m)\top}$$

where $U_r^{(m)} \in \mathbb{R}^{n_m \times r}$ has orthonormal columns: $(U_r^{(m)})^\top U_r^{(m)} = I_r$.

The columns of $U_r^{(m)}$ are the **sample loadings** — each row is a sample's coordinates in the $r$-dimensional subspace that best explains the variance in patch $m$.

### 2.2 The Alignment Step

Given two patches $a$ and $b$ with shared sample set $\mathcal{O} = \mathcal{S}_a \cap \mathcal{S}_b$ (where $|\mathcal{O}| \geq r$), the algorithm solves:

$$\min_G \| U_r^{(b)}[\mathcal{O},:] \cdot G - U_r^{(a)}[\mathcal{O},:] \|_F^2$$

This is a standard least-squares problem. The solution is:

$$G^* = \left( U_r^{(b)}[\mathcal{O},:]^\top U_r^{(b)}[\mathcal{O},:] \right)^{-1} U_r^{(b)}[\mathcal{O},:]^\top \, U_r^{(a)}[\mathcal{O},:]$$

or equivalently the pseudoinverse solution $G^* = U_r^{(b)}[\mathcal{O},:]^+ \, U_r^{(a)}[\mathcal{O},:]$.

### 2.3 What $G$ Represents

If both patches observe the same underlying low-rank structure, then the true relationship is:

$$U_r^{(b)} = U_r^{(a)} Q^\top + E$$

where $Q \in O(r)$ is an orthogonal matrix and $E$ is noise. The least-squares $G^*$ estimates $Q$ (or more precisely, a general linear map that reduces to $Q$ in the noiseless case). **$G$ is not constrained to be orthogonal** — this is a deliberate choice that absorbs scale differences between patches at the cost of potential distortion.

### 2.4 When Alignment Fails

The alignment quality depends on three factors:

1. **Overlap size** $|\mathcal{O}|$: the system $U_b[\mathcal{O},:] G = U_a[\mathcal{O},:]$ has $|\mathcal{O}| \cdot r$ equations and $r^2$ unknowns. We need $|\mathcal{O}| \geq r$ for the system to be determined, but in practice we need $|\mathcal{O}| \gg r$ for stability.

2. **Conditioning of $U_b[\mathcal{O},:]$**: if the overlap samples are near-collinear in the $r$-dimensional subspace (e.g., all from the same cluster), the Gram matrix $U_b[\mathcal{O},:]^\top U_b[\mathcal{O},:]$ is ill-conditioned, and $G$ amplifies noise.

3. **Subspace agreement**: if patches $a$ and $b$ see genuinely different low-rank subspaces (because they observe different features of a structure that is not jointly low-rank), no $G$ can align them well.

In [ ]:
# === Demonstration 2.4: Alignment quality vs. overlap size and conditioning ===

def make_two_patches(N=500, p=50, r=3, K=3, overlap_frac=0.5, seed=42):
    """Create two patches from a shared low-rank structure with known ground truth."""
    rng_local = np.random.default_rng(seed)
    
    # Ground truth: K clusters in r dimensions
    centers = rng_local.standard_normal((K, r)) * 3
    labels = np.repeat(np.arange(K), N // K)
    U_true = centers[labels] + rng_local.standard_normal((len(labels), r)) * 0.3
    
    # Generate full data via random feature loadings
    V1 = rng_local.standard_normal((p, r))
    V2 = rng_local.standard_normal((p, r))
    X1 = U_true @ V1.T + rng_local.standard_normal((len(labels), p)) * 0.1
    X2 = U_true @ V2.T + rng_local.standard_normal((len(labels), p)) * 0.1
    
    # Split samples into two patches with overlap
    n = len(labels)
    mid = n // 2
    ov = int(mid * overlap_frac)
    idx_a = np.arange(0, mid + ov)
    idx_b = np.arange(mid - ov, n)
    
    return X1[idx_a], X2[idx_b], idx_a, idx_b, U_true, labels


def align_patches(Xa, Xb, idx_a, idx_b, r):
    """SVD + least-squares alignment. Returns aligned U for patch b, overlap condition number."""
    Ua, Sa, Vta = np.linalg.svd(Xa, full_matrices=False)
    Ub, Sb, Vtb = np.linalg.svd(Xb, full_matrices=False)
    Ua_r, Ub_r = Ua[:, :r], Ub[:, :r]
    
    # Find overlapping indices
    set_a, set_b = set(idx_a.tolist()), set(idx_b.tolist())
    overlap = sorted(set_a & set_b)
    g2l_a = {g: l for l, g in enumerate(idx_a)}
    g2l_b = {g: l for l, g in enumerate(idx_b)}
    loc_a = [g2l_a[g] for g in overlap]
    loc_b = [g2l_b[g] for g in overlap]
    
    Ua_ov = Ua_r[loc_a]
    Ub_ov = Ub_r[loc_b]
    
    cond = np.linalg.cond(Ub_ov)
    G, _, _, _ = np.linalg.lstsq(Ub_ov, Ua_ov, rcond=None)
    Ub_aligned = Ub_r @ G
    
    # Measure alignment error on overlap
    resid = np.linalg.norm(Ub_aligned[loc_b] - Ua_r[loc_a], 'fro')
    return Ua_r, Ub_aligned, G, cond, resid, overlap, loc_a, loc_b


# Sweep overlap fraction
overlaps = [0.02, 0.05, 0.1, 0.2, 0.4, 0.8]
conds, resids, n_ovs = [], [], []

for ov in overlaps:
    Xa, Xb, ia, ib, U_true, labels = make_two_patches(overlap_frac=ov)
    _, _, G, c, res, overlap, _, _ = align_patches(Xa, Xb, ia, ib, r=3)
    conds.append(c)
    resids.append(res)
    n_ovs.append(len(overlap))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].semilogy(overlaps, conds, 'o-', color='#d62728')
axes[0].set_xlabel('Overlap fraction')
axes[0].set_ylabel('Condition number of $U_b[\\mathcal{O},:]$')
axes[0].set_title('Overlap conditioning')
axes[0].axhline(1e4, ls='--', color='gray', alpha=0.5, label='Ill-conditioned threshold')
axes[0].legend(fontsize=8)

axes[1].plot(overlaps, resids, 's-', color='#2ca02c')
axes[1].set_xlabel('Overlap fraction')
axes[1].set_ylabel('$\\|U_b G - U_a\\|_F$ on overlap')
axes[1].set_title('Alignment residual')

axes[2].bar(range(len(overlaps)), n_ovs, color='#1f77b4', alpha=0.7)
axes[2].set_xticks(range(len(overlaps)))
axes[2].set_xticklabels([f'{o:.0%}' for o in overlaps])
axes[2].set_xlabel('Overlap fraction')
axes[2].set_ylabel('Number of shared samples')
axes[2].set_title('Overlap size')
axes[2].axhline(3, ls='--', color='red', alpha=0.5, label='$r = 3$ (minimum)')
axes[2].legend(fontsize=8)

fig.suptitle('Alignment Quality Degrades with Insufficient Overlap', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='3'></a>
## 3. Sequential Quilting: Error Propagation Along the Chain

### 3.1 The Sequential Chain

In Algorithm 1 (sequential quilting), patches are ordered into a chain $P_1, P_2, \ldots, P_M$ by greedy maximum overlap. Alignment proceeds left-to-right:

1. $\widetilde{U} \leftarrow U_r^{(1)}$ (first patch initializes the global embedding)
2. For $m = 2, \ldots, M$: solve $U_r^{(m)}[\mathcal{O}_m,:] \, G_m \approx \widetilde{U}[\mathcal{O}_m,:]$, then set $\widetilde{U}[\text{new}_m,:] = U_r^{(m)}[\text{new}_m,:] \, G_m$

### 3.2 Error Compounding

Each alignment $G_m$ is estimated with error $\Delta G_m$. The aligned embedding for new samples in patch $m$ is:

$$\widetilde{U}_{\text{new}_m} = U_r^{(m)}_{\text{new}_m} (G_m^* + \Delta G_m)$$

But the target $\widetilde{U}[\mathcal{O}_m,:]$ that $G_m$ aligns to was *itself* the product of earlier alignments. For the overlap samples that came from patch $m-1$, the target already contains the error from $G_{m-1}$.

Unrolling the recursion for a sample that first appears in patch $k$ and whose overlap predecessors trace back through patches $k-1, k-2, \ldots, 1$:

$$\widetilde{U}_{\text{sample}} = U_r^{(k)} \cdot (G_k^* + \Delta G_k) \approx U_r^{(k)} \cdot G_k^* \cdot (I + G_k^{*-1} \Delta G_k)$$

The effective transformation accumulates:

$$G_{\text{eff}} = \prod_{j=2}^{k} (G_j^* + \Delta G_j) = \prod_{j=2}^{k} G_j^* \cdot \prod_{j=2}^{k} (I + G_j^{*-1} \Delta G_j)$$

The error product $\prod_{j=2}^{k} (I + E_j)$ expands as:

$$(I + E_2)(I + E_3)\cdots(I + E_k) = I + \sum_j E_j + \sum_{j<l} E_j E_l + \cdots$$

To first order, the total error grows as $\sum_{j=2}^{k} \|E_j\|$, i.e., **linearly with chain length** $k$. This is the $O(M)$ error accumulation of sequential quilting.

### 3.3 The Critical Vulnerability

An early bad alignment (large $\|\Delta G_2\|$) propagates through *every* subsequent alignment, because all later patches align against a target that already contains this error. This is why:

- More patches → more chain steps → more compounding
- Low overlap → larger $\|\Delta G_m\|$ per step → faster divergence
- A single catastrophic alignment can corrupt the entire downstream embedding

<a id='4'></a>
## 4. Hierarchical Quilting: The Promise

### 4.1 Motivation

The hierarchical variant replaces the $O(M)$-depth chain with an $O(\log_2 M)$-depth binary tree:

```
Level 0 (leaves):   P1   P2   P3   P4   P5   P6   P7   P8
Level 1 (merges):   [P1,P2]  [P3,P4]  [P5,P6]  [P7,P8]
Level 2:            [P1-P4]          [P5-P8]
Level 3 (root):     [P1-P8]
```

At each level, nodes are paired by greedy maximum sample overlap, and the smaller node is aligned onto the larger.

### 4.2 Theoretical Advantages

**Reduced chain depth**: any sample passes through at most $\lceil \log_2 M \rceil$ alignments instead of $M - 1$. By the same first-order error analysis:

$$\text{Total error} \sim \sum_{j=1}^{\log_2 M} \|E_j\| \quad \text{vs.} \quad \sum_{j=1}^{M-1} \|E_j\|$$

**Error isolation**: a bad merge at one branch does not propagate to the other branch. In the sequential chain, a bad $G_3$ corrupts patches 4 through $M$. In the tree, a bad merge of $[P_1, P_2]$ only affects the subtree containing $P_1$ and $P_2$.

**Larger overlap at merges**: by the time two branches are merged at a higher level, each branch covers many samples. If the covered sample sets overlap, the merge has a large overlap set, improving conditioning.

### 4.3 Why This Should Work

For $M = 12$ patches:
- Sequential: up to 11 alignment steps in a chain ($O(M)$)
- Hierarchical: up to 4 levels ($\lceil \log_2 12 \rceil = 4$), so $O(\log M)$ alignment depth

If per-step error is similar, the tree should reduce total error by a factor of $\sim M / \log_2 M \approx 3\times$ for $M = 12$.

<a id='5'></a>
## 5. Hierarchical Quilting: The Failure — Greedy Locality → Global Separation

### 5.1 The Greedy Pairing Problem

The hierarchical algorithm pairs nodes at each level by **greedy maximum sample overlap**. This seems reasonable — high overlap means better alignment. But it creates a structural problem.

Consider patches arranged along the sample axis:

```
Samples:  [=====P1=====][=====P2=====][=====P3=====][=====P4=====]
Overlap:        ^^^            ^^^            ^^^
```

Each patch overlaps most with its **neighbors**. Greedy pairing merges:
- Level 1: $(P_1, P_2)$ and $(P_3, P_4)$ — both high-overlap pairs
- Level 2: $([P_1, P_2], [P_3, P_4])$ — but these two super-nodes may share very few samples!

### 5.2 The Global Separation Failure Mode

The greedy strategy is **locally optimal but globally catastrophic**:

1. **Level 1 merges are excellent**: adjacent patches share many samples, alignment is well-conditioned.

2. **Higher-level merges are starved**: after merging adjacent pairs, the resulting super-nodes cover contiguous, non-overlapping sample ranges. The overlap between the left half $[P_1, \ldots, P_{M/2}]$ and right half $[P_{M/2+1}, \ldots, P_M]$ comes only from the boundary patches $P_{M/2}$ and $P_{M/2+1}$ — which have already been absorbed into their respective branches.

3. **The final merge is the weakest link**: the root merge must bridge two halves of the sample space with whatever boundary overlap remains. If this overlap is small or ill-conditioned, the entire tree alignment degrades.

### 5.3 Formal Analysis: Overlap Attenuation

Let $o_{ij} = |\mathcal{S}_i \cap \mathcal{S}_j|$ denote the sample overlap between patches $i$ and $j$. For patches tiling the sample axis with overlap fraction $f$:

$$o_{i,i+1} \approx f \cdot n_i \quad \text{(adjacent overlap)}$$
$$o_{i,j} = 0 \quad \text{for } |i - j| \geq 2 \quad \text{(non-adjacent patches don't overlap)}$$

After greedy pairing at level 1:
- Super-node $A = [P_1, P_2]$ covers samples $\mathcal{S}_1 \cup \mathcal{S}_2$
- Super-node $B = [P_3, P_4]$ covers samples $\mathcal{S}_3 \cup \mathcal{S}_4$

The overlap $|\mathcal{S}_A \cap \mathcal{S}_B| = |\mathcal{S}_2 \cap \mathcal{S}_3| = o_{2,3} \approx f \cdot n$ — this is the same as a single patch-pair overlap. But now this overlap must align a node containing $\sim 2n$ samples. The **overlap-to-node-size ratio** shrinks by half at each tree level:

$$\text{Level } \ell: \quad \frac{|\text{overlap}|}{|\text{node size}|} \approx \frac{f \cdot n}{2^\ell \cdot n} = \frac{f}{2^\ell}$$

By the root merge ($\ell = \log_2 M - 1$), the effective overlap fraction is $\sim f / (M/2)$, which can be dangerously small.

### 5.4 Why Sequential Doesn't Have This Problem

In the sequential chain, each new patch aligns against the **entire accumulated embedding** $\widetilde{U}$. The overlap set is $\mathcal{S}_m \cap \bigcup_{j<m} \mathcal{S}_j$, which includes *all* previously seen samples in patch $m$'s range. The greedy ordering ensures this overlap is maximized at each step.

The sequential chain distributes alignment work *evenly* across all $M - 1$ steps. Each step aligns one patch against everything seen so far. Even though errors compound, each individual alignment is well-conditioned because the overlap is always between the new patch and the large accumulated set.

The hierarchical tree **front-loads** easy merges (adjacent patches) and **back-loads** the hardest merge (the two halves). This is the opposite of what error theory suggests: the highest-level merge, which affects the most samples, should have the *best* alignment quality, not the worst.

<a id='6'></a>
## 6. Demonstration: SVD Alignment Failure Modes

We construct a minimal example showing how the least-squares alignment $G$ degrades as (a) overlap shrinks and (b) overlap samples become cluster-degenerate (all from one cluster).

In [ ]:
# === 6.1 Minimal alignment example: 3 clusters in 2D ===

def make_clustered_data(N=300, r=2, K=3, separation=5, seed=0):
    """Generate clustered data with known structure."""
    rng_l = np.random.default_rng(seed)
    n_per = N // K
    labels = np.repeat(np.arange(K), n_per)
    
    # Cluster centers well-separated in r dimensions
    angles = np.linspace(0, 2 * np.pi, K, endpoint=False)
    centers = np.column_stack([np.cos(angles), np.sin(angles)]) * separation
    
    U_true = centers[labels] + rng_l.standard_normal((len(labels), r)) * 0.5
    return U_true, labels, centers


U_true, labels, centers = make_clustered_data()
N = len(labels)

# Two patches: A gets first 200 samples, B gets last 200 samples
# Overlap region: samples 100-200
idx_a = np.arange(0, 200)
idx_b = np.arange(100, 300)

# Generate observed data through random projections (different features per patch)
p = 20
V_a = rng.standard_normal((p, 2))
V_b = rng.standard_normal((p, 2))
X_a = U_true[idx_a] @ V_a.T + rng.standard_normal((200, p)) * 0.1
X_b = U_true[idx_b] @ V_b.T + rng.standard_normal((200, p)) * 0.1

# SVD of each patch
Ua, Sa, Vta = np.linalg.svd(X_a, full_matrices=False)
Ub, Sb, Vtb = np.linalg.svd(X_b, full_matrices=False)
Ua2 = Ua[:, :2]
Ub2 = Ub[:, :2]

# Align B onto A using full overlap (100 samples)
# Overlap: global indices 100-199. In A: local 100-199. In B: local 0-99.
overlap_a = np.arange(100, 200)  # local indices in A
overlap_b = np.arange(0, 100)    # local indices in B

G_full, _, _, _ = np.linalg.lstsq(Ub2[overlap_b], Ua2[overlap_a], rcond=None)
Ub2_aligned_full = Ub2 @ G_full

# Now try with tiny overlap (5 samples)
G_tiny, _, _, _ = np.linalg.lstsq(Ub2[overlap_b[:5]], Ua2[overlap_a[:5]], rcond=None)
Ub2_aligned_tiny = Ub2 @ G_tiny

# Try with single-cluster overlap (only cluster 1 samples in overlap)
cluster1_mask_b = labels[idx_b][overlap_b] == 1
cluster1_ov_b = overlap_b[cluster1_mask_b]
cluster1_ov_a = overlap_a[cluster1_mask_b]
G_degen, _, _, _ = np.linalg.lstsq(Ub2[cluster1_ov_b], Ua2[cluster1_ov_a], rcond=None)
Ub2_aligned_degen = Ub2 @ G_degen


fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
colors = ['#e41a1c', '#377eb8', '#4daf4a']

# Panel 1: Unaligned
for k in range(3):
    mask_a = labels[idx_a] == k
    mask_b = labels[idx_b] == k
    axes[0].scatter(Ua2[mask_a, 0], Ua2[mask_a, 1], c=colors[k], s=8, alpha=0.5, marker='o')
    axes[0].scatter(Ub2[mask_b, 0], Ub2[mask_b, 1], c=colors[k], s=8, alpha=0.5, marker='^')
axes[0].set_title('Unaligned SVD bases\n(circles=A, triangles=B)')

# Panel 2: Full overlap alignment (100 samples)
for k in range(3):
    mask_a = labels[idx_a] == k
    mask_b = labels[idx_b] == k
    axes[1].scatter(Ua2[mask_a, 0], Ua2[mask_a, 1], c=colors[k], s=8, alpha=0.5, marker='o')
    axes[1].scatter(Ub2_aligned_full[mask_b, 0], Ub2_aligned_full[mask_b, 1],
                    c=colors[k], s=8, alpha=0.5, marker='^')
axes[1].set_title(f'Aligned (100 overlap samples)\ncond={np.linalg.cond(Ub2[overlap_b]):.1f}')

# Panel 3: Tiny overlap (5 samples)
for k in range(3):
    mask_a = labels[idx_a] == k
    mask_b = labels[idx_b] == k
    axes[2].scatter(Ua2[mask_a, 0], Ua2[mask_a, 1], c=colors[k], s=8, alpha=0.5, marker='o')
    axes[2].scatter(Ub2_aligned_tiny[mask_b, 0], Ub2_aligned_tiny[mask_b, 1],
                    c=colors[k], s=8, alpha=0.5, marker='^')
axes[2].set_title(f'Aligned (5 overlap samples)\ncond={np.linalg.cond(Ub2[overlap_b[:5]]):.1f}')

# Panel 4: Single-cluster overlap
for k in range(3):
    mask_a = labels[idx_a] == k
    mask_b = labels[idx_b] == k
    axes[3].scatter(Ua2[mask_a, 0], Ua2[mask_a, 1], c=colors[k], s=8, alpha=0.5, marker='o')
    axes[3].scatter(Ub2_aligned_degen[mask_b, 0], Ub2_aligned_degen[mask_b, 1],
                    c=colors[k], s=8, alpha=0.5, marker='^')
axes[3].set_title(f'Aligned (single-cluster overlap)\ncond={np.linalg.cond(Ub2[cluster1_ov_b]):.1f}')

for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')

fig.suptitle('SVD Alignment Failure Modes', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'G (full overlap):\n{G_full}')
print(f'\nG (5 samples):\n{G_tiny}')
print(f'\nG (single-cluster):\n{G_degen}')
print(f'\n||G_full|| = {np.linalg.norm(G_full):.3f}')
print(f'||G_tiny|| = {np.linalg.norm(G_tiny):.3f}  (should be similar to full)')
print(f'||G_degen|| = {np.linalg.norm(G_degen):.3f}  (likely inflated — ill-conditioned)')

### 6.2 Interpretation

The four panels above show the same two patches under different alignment conditions:

1. **Unaligned**: the SVD bases are in arbitrary orientations — clusters don't match across patches.
2. **Full overlap (100 samples)**: alignment is excellent — clusters merge seamlessly.
3. **Tiny overlap (5 samples)**: the least-squares system is barely determined ($|\mathcal{O}| = 5$ for $r^2 = 4$ unknowns). $G$ may overfit the 5 points and distort the rest.
4. **Single-cluster overlap**: all overlap samples come from one cluster, so $U_b[\mathcal{O},:]$ has near-rank-1 structure. The Gram matrix is ill-conditioned, and $G$ can only recover the alignment along one direction — the other direction is arbitrary.

Both failure modes (3 and 4) are **exactly what happens at high-level merges in hierarchical quilting**: the overlap is small, and if the boundary patches happen to be cluster-homogeneous, the overlap is degenerate.

<a id='7'></a>
## 7. Demonstration: Chain vs. Tree Error Accumulation

We simulate pure alignment error propagation (no real data) to compare the theoretical error behavior of sequential vs. hierarchical merging.

In [ ]:
# === 7.1 Abstract error propagation simulation ===
#
# Model: each alignment introduces a random rotation error.
# We track the total accumulated rotation away from ground truth.

def random_rotation_2d(angle_std, rng):
    """Random 2D rotation matrix with angle ~ N(0, angle_std^2)."""
    theta = rng.normal(0, angle_std)
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def sequential_error(M, angle_std, rng):
    """Simulate sequential chain: each step applies G_true + noise."""
    # Ground truth: all patches should map to identity
    accumulated = np.eye(2)  # patch 1 is reference
    errors = [0.0]  # patch 1 has zero error
    
    for m in range(1, M):
        # Noisy alignment: G = I + perturbation
        noise = random_rotation_2d(angle_std, rng)
        accumulated = accumulated @ noise  # error compounds
        # Angular error from identity
        angle_err = np.abs(np.arctan2(accumulated[1, 0], accumulated[0, 0]))
        errors.append(np.degrees(angle_err))
    
    return errors


def hierarchical_error_uniform(M, angle_std, rng):
    """Simulate hierarchical tree: each level merges pairs with error.
    Assumes uniform overlap quality at all levels (best case for tree)."""
    # Each leaf has identity transform
    transforms = [np.eye(2) for _ in range(M)]
    
    while len(transforms) > 1:
        next_level = []
        for i in range(0, len(transforms) - 1, 2):
            # Anchor = left, align right onto left with noise
            noise = random_rotation_2d(angle_std, rng)
            # Right node picks up one more error
            aligned_right = transforms[i + 1] @ noise
            # Merged node: left keeps its transform, right gets additional error
            # For tracking, store the "worst" transform in the merged node
            next_level.append(aligned_right)  # right has more error
        if len(transforms) % 2 == 1:
            next_level.append(transforms[-1])
        transforms = next_level
    
    # The max error at any leaf after tree alignment:
    # depth = ceil(log2(M)), each level adds one error
    # Return the error at each depth level
    depth = int(np.ceil(np.log2(M)))
    cum_errors = [0.0]
    acc = np.eye(2)
    for d in range(depth):
        noise = random_rotation_2d(angle_std, rng)
        acc = acc @ noise
        angle_err = np.abs(np.arctan2(acc[1, 0], acc[0, 0]))
        cum_errors.append(np.degrees(angle_err))
    return cum_errors


def hierarchical_error_degrading(M, angle_std_base, rng):
    """Simulate hierarchical tree with DEGRADING overlap at higher levels.
    This is the realistic case: higher-level merges have worse overlap."""
    depth = int(np.ceil(np.log2(M)))
    cum_errors = [0.0]
    acc = np.eye(2)
    for d in range(depth):
        # Error grows at higher levels due to overlap attenuation
        level_noise_std = angle_std_base * (2 ** d)  # doubles each level
        noise = random_rotation_2d(level_noise_std, rng)
        acc = acc @ noise
        angle_err = np.abs(np.arctan2(acc[1, 0], acc[0, 0]))
        cum_errors.append(np.degrees(angle_err))
    return cum_errors


# Run many trials
M = 12
angle_std = 0.05  # ~3 degrees per step
n_trials = 500

seq_errors = np.array([sequential_error(M, angle_std, np.random.default_rng(s))
                       for s in range(n_trials)])
tree_uniform = np.array([hierarchical_error_uniform(M, angle_std, np.random.default_rng(s))
                         for s in range(n_trials)])
tree_degrade = np.array([hierarchical_error_degrading(M, angle_std, np.random.default_rng(s))
                         for s in range(n_trials)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: sequential vs tree (uniform overlap)
ax = axes[0]
x_seq = np.arange(M)
x_tree = np.arange(tree_uniform.shape[1])
ax.plot(x_seq, seq_errors.mean(axis=0), 'o-', color='#d62728', label='Sequential (chain)', lw=2)
ax.fill_between(x_seq, seq_errors.mean(0) - seq_errors.std(0),
                seq_errors.mean(0) + seq_errors.std(0), color='#d62728', alpha=0.15)
ax.plot(x_tree, tree_uniform.mean(axis=0), 's-', color='#2ca02c',
        label='Hierarchical (uniform overlap)', lw=2)
ax.fill_between(x_tree, tree_uniform.mean(0) - tree_uniform.std(0),
                tree_uniform.mean(0) + tree_uniform.std(0), color='#2ca02c', alpha=0.15)
ax.set_xlabel('Alignment step (sequential) / Tree depth (hierarchical)')
ax.set_ylabel('Accumulated angular error (degrees)')
ax.set_title('Ideal case: uniform overlap quality\n(tree wins as expected)')
ax.legend()
ax.grid(alpha=0.3)

# Right: sequential vs tree (degrading overlap)
ax = axes[1]
ax.plot(x_seq, seq_errors.mean(axis=0), 'o-', color='#d62728', label='Sequential (chain)', lw=2)
ax.fill_between(x_seq, seq_errors.mean(0) - seq_errors.std(0),
                seq_errors.mean(0) + seq_errors.std(0), color='#d62728', alpha=0.15)
ax.plot(x_tree, tree_degrade.mean(axis=0), '^-', color='#ff7f0e',
        label='Hierarchical (degrading overlap)', lw=2)
ax.fill_between(x_tree, tree_degrade.mean(0) - tree_degrade.std(0),
                tree_degrade.mean(0) + tree_degrade.std(0), color='#ff7f0e', alpha=0.15)
ax.set_xlabel('Alignment step (sequential) / Tree depth (hierarchical)')
ax.set_ylabel('Accumulated angular error (degrees)')
ax.set_title('Realistic case: overlap degrades at higher levels\n(tree loses despite fewer steps)')
ax.legend()
ax.grid(alpha=0.3)

fig.suptitle(f'Error Accumulation: Chain O(M) vs. Tree O(log M), M={M} patches',
             fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print(f'Sequential: {M-1} alignment steps, final error = {seq_errors[:, -1].mean():.1f} +/- {seq_errors[:, -1].std():.1f} deg')
print(f'Tree (uniform): {tree_uniform.shape[1]-1} levels, final error = {tree_uniform[:, -1].mean():.1f} +/- {tree_uniform[:, -1].std():.1f} deg')
print(f'Tree (degrading): {tree_degrade.shape[1]-1} levels, final error = {tree_degrade[:, -1].mean():.1f} +/- {tree_degrade[:, -1].std():.1f} deg')

### 7.2 Interpretation

**Left panel (ideal case)**: When overlap quality is uniform across all tree levels, the hierarchical tree achieves its theoretical $O(\log M)$ error — substantially less than the sequential chain's $O(M)$ growth. This is the scenario the hierarchical method was designed for.

**Right panel (realistic case)**: When overlap quality degrades at higher tree levels (as it does with greedy pairing of spatially contiguous patches), the hierarchical tree can accumulate *more* total error than the sequential chain, despite having fewer alignment steps. The per-step error at the root merge is so large that it overwhelms the depth advantage.

The key insight: **the hierarchical tree's $O(\log M)$ depth advantage is only realized when per-step error is roughly constant across levels**. Greedy pairing violates this assumption catastrophically.

<a id='8'></a>
## 8. Demonstration: The Global Separation Problem

We now construct a concrete example showing the full pipeline: patch generation, sequential vs. hierarchical quilting, and how greedy pairing leads to a starved root merge.

In [ ]:
# === 8.1 Generate synthetic patchwork data ===

def generate_patchwork(N=600, p=60, K=3, r=3, n_patches=8, overlap_frac=0.2, seed=42):
    """Generate clustered data split into patches with controlled overlap."""
    rng_l = np.random.default_rng(seed)
    
    # Ground truth clusters
    labels = np.repeat(np.arange(K), N // K)
    centers = rng_l.standard_normal((K, r)) * 5
    U_true = centers[labels] + rng_l.standard_normal((N, r)) * 0.5
    
    # Full data
    V_full = rng_l.standard_normal((p, r))
    X_full = U_true @ V_full.T + rng_l.standard_normal((N, p)) * 0.1
    
    # Create patches tiling sample axis with overlap
    core = N // n_patches
    ov = int(core * overlap_frac)
    
    # Feature blocks: tile feature axis
    feat_per_patch = p // n_patches + 5  # some overlap in features too
    
    patches = []
    for i in range(n_patches):
        row_start = max(0, i * core - ov)
        row_end = min(N, (i + 1) * core + ov)
        col_start = (i * (p // n_patches)) % p
        col_end = min(p, col_start + feat_per_patch)
        
        patches.append({
            'row_idx': np.arange(row_start, row_end),
            'col_idx': np.arange(col_start, col_end),
        })
    
    return X_full, U_true, labels, patches


X_full, U_true, labels, patches = generate_patchwork(n_patches=8, overlap_frac=0.2)
n_patches = len(patches)

# Visualize patch structure
fig, ax = plt.subplots(figsize=(10, 4))
colors_patch = plt.cm.Set3(np.linspace(0, 1, n_patches))

for i, p in enumerate(patches):
    r_min, r_max = p['row_idx'][0], p['row_idx'][-1]
    c_min, c_max = p['col_idx'][0], p['col_idx'][-1]
    rect = plt.Rectangle((c_min, r_min), c_max - c_min, r_max - r_min,
                          fill=True, facecolor=colors_patch[i], edgecolor='black',
                          alpha=0.5, lw=1.5)
    ax.add_patch(rect)
    ax.text((c_min + c_max) / 2, (r_min + r_max) / 2, f'P{i+1}',
            ha='center', va='center', fontweight='bold', fontsize=9)

ax.set_xlim(-1, 61)
ax.set_ylim(-10, 610)
ax.invert_yaxis()
ax.set_xlabel('Features')
ax.set_ylabel('Samples')
ax.set_title('Patch Architecture: 8 patches, 20% overlap\nNote: adjacent patches overlap in samples, distant patches do not')
plt.tight_layout()
plt.show()

# Print overlap matrix
print('Sample overlap matrix (|S_i ∩ S_j|):')
ov_mat = np.zeros((n_patches, n_patches), dtype=int)
for i in range(n_patches):
    for j in range(n_patches):
        ov_mat[i, j] = len(set(patches[i]['row_idx'].tolist()) & set(patches[j]['row_idx'].tolist()))
print(ov_mat)

In [ ]:
# === 8.2 Sequential quilting ===

def greedy_ordering(patches):
    """Greedy patch ordering by max overlap with accumulated set."""
    n = len(patches)
    row_sets = [set(p['row_idx'].tolist()) for p in patches]
    
    best_score, best_pair = -1, (0, 1)
    for i in range(n):
        for j in range(i + 1, n):
            s = len(row_sets[i] & row_sets[j])
            if s > best_score:
                best_score, best_pair = s, (i, j)
    
    ordering = [best_pair[0], best_pair[1]]
    used = set(ordering)
    acc = row_sets[best_pair[0]] | row_sets[best_pair[1]]
    
    for _ in range(2, n):
        best_next, best_s = -1, -1
        for j in range(n):
            if j not in used:
                s = len(row_sets[j] & acc)
                if s > best_s:
                    best_s, best_next = s, j
        ordering.append(best_next)
        used.add(best_next)
        acc |= row_sets[best_next]
    return ordering


def sequential_quilt(patches, X_full, r=3, K=3):
    """Sequential quilting (Algorithm 1)."""
    M, N_feat = X_full.shape
    U_tilde = np.zeros((M, r))
    covered = set()
    ordering = greedy_ordering(patches)
    align_conds = []  # track conditioning at each step
    
    m0 = ordering[0]
    ri, ci = patches[m0]['row_idx'], patches[m0]['col_idx']
    U, S, Vt = np.linalg.svd(X_full[np.ix_(ri, ci)], full_matrices=False)
    U_tilde[ri] = U[:, :r]
    covered.update(ri.tolist())
    
    for step in range(1, len(ordering)):
        m = ordering[step]
        ri, ci = patches[m]['row_idx'], patches[m]['col_idx']
        U, S, Vt = np.linalg.svd(X_full[np.ix_(ri, ci)], full_matrices=False)
        U_r = U[:, :r]
        
        cur = set(ri.tolist())
        ov = sorted(cur & covered)
        
        if len(ov) < r:
            U_tilde[ri] = U_r
            covered.update(cur)
            align_conds.append(np.inf)
            continue
        
        g2l = {g: l for l, g in enumerate(ri)}
        loc_ov = [g2l[g] for g in ov]
        
        cond = np.linalg.cond(U_r[loc_ov])
        align_conds.append(cond)
        
        G, _, _, _ = np.linalg.lstsq(U_r[loc_ov], U_tilde[ov], rcond=None)
        
        new = sorted(cur - covered)
        if new:
            new_loc = [g2l[g] for g in new]
            U_tilde[new] = U_r[new_loc] @ G
        covered.update(cur)
    
    km = KMeans(n_clusters=K, n_init=20, random_state=42)
    pred = km.fit_predict(U_tilde)
    return pred, U_tilde, align_conds, ordering


pred_seq, U_seq, conds_seq, ordering = sequential_quilt(patches, X_full)
ari_seq = adjusted_rand_score(labels, pred_seq)
print(f'Sequential ARI: {ari_seq:.3f}')
print(f'Alignment condition numbers: {[f"{c:.0f}" for c in conds_seq]}')
print(f'Patch ordering: {ordering}')

In [ ]:
# === 8.3 Hierarchical quilting ===

def greedy_pairing(nodes):
    """Pair nodes by max overlap."""
    n = len(nodes)
    if n == 1:
        return [], 0
    row_sets = [set(nd['rows'].tolist()) for nd in nodes]
    scored = []
    for i in range(n):
        for j in range(i + 1, n):
            scored.append((len(row_sets[i] & row_sets[j]), i, j))
    scored.sort(reverse=True)
    used = set()
    pairs = []
    for _, i, j in scored:
        if i in used or j in used:
            continue
        pairs.append((i, j))
        used.update([i, j])
    unpaired = None
    for k in range(n):
        if k not in used:
            unpaired = k
    return pairs, unpaired


def merge_nodes(a, b, r):
    """Merge node b onto a via shared samples."""
    rows_a = set(a['rows'].tolist())
    rows_b = set(b['rows'].tolist())
    ov = sorted(rows_a & rows_b)
    
    g2l_a = {g: l for l, g in enumerate(a['rows'])}
    g2l_b = {g: l for l, g in enumerate(b['rows'])}
    
    merge_cond = np.nan
    if len(ov) >= r:
        la = [g2l_a[g] for g in ov]
        lb = [g2l_b[g] for g in ov]
        merge_cond = np.linalg.cond(b['U'][lb])
        G, _, _, _ = np.linalg.lstsq(b['U'][lb], a['U'][la], rcond=None)
        Ub_aligned = b['U'] @ G
    else:
        Ub_aligned = b['U']
    
    new_in_b = sorted(rows_b - rows_a)
    union = list(a['rows']) + new_in_b
    U_merged = np.empty((len(union), r))
    U_merged[:len(a['rows'])] = a['U']
    if new_in_b:
        new_lb = [g2l_b[g] for g in new_in_b]
        U_merged[len(a['rows']):] = Ub_aligned[new_lb]
    
    return {'U': U_merged, 'rows': np.array(union, dtype=np.int64)}, len(ov), merge_cond


def hierarchical_quilt(patches, X_full, r=3, K=3):
    """Hierarchical quilting with merge tracking."""
    nodes = []
    for p in patches:
        ri, ci = p['row_idx'], p['col_idx']
        U, S, Vt = np.linalg.svd(X_full[np.ix_(ri, ci)], full_matrices=False)
        nodes.append({'U': U[:, :r].copy(), 'rows': np.array(ri, dtype=np.int64)})
    
    merge_log = []  # (level, overlap_size, condition_number, node_sizes)
    level = 0
    
    while len(nodes) > 1:
        pairs, unpaired = greedy_pairing(nodes)
        next_level = []
        for i, j in pairs:
            if len(nodes[i]['rows']) >= len(nodes[j]['rows']):
                merged, ov_sz, cond = merge_nodes(nodes[i], nodes[j], r)
            else:
                merged, ov_sz, cond = merge_nodes(nodes[j], nodes[i], r)
            merge_log.append((level, ov_sz, cond, len(nodes[i]['rows']), len(nodes[j]['rows'])))
            next_level.append(merged)
        if unpaired is not None:
            next_level.append(nodes[unpaired])
        nodes = next_level
        level += 1
    
    # Extract final U_tilde
    final = nodes[0]
    M = X_full.shape[0]
    U_tilde = np.zeros((M, r))
    for idx_local, idx_global in enumerate(final['rows']):
        U_tilde[idx_global] = final['U'][idx_local]
    
    km = KMeans(n_clusters=K, n_init=20, random_state=42)
    pred = km.fit_predict(U_tilde)
    return pred, U_tilde, merge_log


pred_hier, U_hier, merge_log = hierarchical_quilt(patches, X_full)
ari_hier = adjusted_rand_score(labels, pred_hier)
print(f'Hierarchical ARI: {ari_hier:.3f}')
print(f'\nMerge log (level, overlap_size, condition_number, node_a_size, node_b_size):')
for entry in merge_log:
    lvl, ov, cond, na, nb = entry
    print(f'  Level {lvl}: overlap={ov:3d}, cond={cond:10.1f}, sizes=({na}, {nb})')

In [ ]:
# === 8.4 Visualize the merge tree overlap degradation ===

levels = [e[0] for e in merge_log]
ov_sizes = [e[1] for e in merge_log]
cond_nums = [e[2] for e in merge_log]
node_sizes = [e[3] + e[4] for e in merge_log]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Overlap size per merge
for i, (lvl, ov, cond, na, nb) in enumerate(merge_log):
    axes[0].bar(i, ov, color=plt.cm.viridis(lvl / max(levels + [1])), edgecolor='black')
    axes[0].text(i, ov + 1, f'L{lvl}', ha='center', fontsize=8)
axes[0].set_xlabel('Merge index')
axes[0].set_ylabel('Overlap size')
axes[0].set_title('Shared samples per merge')
axes[0].axhline(3, ls='--', color='red', alpha=0.5, label='$r = 3$')
axes[0].legend(fontsize=8)

# Condition number per merge
for i, (lvl, ov, cond, na, nb) in enumerate(merge_log):
    axes[1].bar(i, cond, color=plt.cm.viridis(lvl / max(levels + [1])), edgecolor='black')
    axes[1].text(i, cond * 1.05, f'L{lvl}', ha='center', fontsize=8)
axes[1].set_xlabel('Merge index')
axes[1].set_ylabel('Condition number')
axes[1].set_title('$U_b[\\mathcal{O},:]$ conditioning per merge')
axes[1].set_yscale('log')

# Overlap / node size ratio
for i, (lvl, ov, cond, na, nb) in enumerate(merge_log):
    ratio = ov / (na + nb)
    axes[2].bar(i, ratio, color=plt.cm.viridis(lvl / max(levels + [1])), edgecolor='black')
    axes[2].text(i, ratio + 0.005, f'L{lvl}', ha='center', fontsize=8)
axes[2].set_xlabel('Merge index')
axes[2].set_ylabel('Overlap / merged node size')
axes[2].set_title('Effective overlap fraction per merge\n(degrades at higher levels)')

fig.suptitle('Hierarchical Merge Quality Degrades at Higher Tree Levels', fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# === 8.5 Compare embeddings ===

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_cluster = ['#e41a1c', '#377eb8', '#4daf4a']

# Ground truth
for k in range(3):
    mask = labels == k
    axes[0].scatter(U_true[mask, 0], U_true[mask, 1], c=colors_cluster[k], s=6, alpha=0.5)
axes[0].set_title('Ground truth embedding')

# Sequential
for k in range(3):
    mask = pred_seq == k
    axes[1].scatter(U_seq[mask, 0], U_seq[mask, 1], c=colors_cluster[k], s=6, alpha=0.5)
axes[1].set_title(f'Sequential quilting (ARI={ari_seq:.3f})')

# Hierarchical
for k in range(3):
    mask = pred_hier == k
    axes[2].scatter(U_hier[mask, 0], U_hier[mask, 1], c=colors_cluster[k], s=6, alpha=0.5)
axes[2].set_title(f'Hierarchical quilting (ARI={ari_hier:.3f})')

for ax in axes:
    ax.set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')

fig.suptitle('Embedding Comparison: Sequential vs. Hierarchical', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 8.6 Anatomy of the Root Merge Failure

The merge log above reveals the core problem. Examine the **overlap** and **condition number** columns:

- **Level 0** (leaf merges): adjacent patches share many samples. The overlap sets are large, $U_b[\mathcal{O},:]$ is well-conditioned, and the least-squares $G$ is accurate.

- **Level 1+** (higher merges): the super-nodes cover contiguous, largely disjoint sample ranges. The overlap comes only from the thin boundary between the two halves. The overlap-to-node-size ratio drops precipitously.

- **Root merge**: the two largest super-nodes must align using only the samples shared between their respective boundary patches. This is a tiny fraction of the total samples, leading to either:
  - **Insufficient overlap** ($|\mathcal{O}| < r$): alignment is skipped entirely, leaving half the embedding unaligned.
  - **Ill-conditioned overlap**: $G$ amplifies noise, distorting the aligned half.

This is the **"greedy locality → global separation"** failure: the greedy pairing strategy produces locally excellent merges but guarantees that the globally critical root merge is the worst-conditioned.

In [ ]:
# === 8.7 Sweep: how the gap grows with patch count ===

patch_counts = [2, 3, 4, 6, 8, 10, 12, 14]
aris_seq = []
aris_hier = []
n_trials_sweep = 10

for M in patch_counts:
    seq_vals, hier_vals = [], []
    for trial in range(n_trials_sweep):
        X_f, U_t, lbls, pats = generate_patchwork(
            N=600, p=60, K=3, n_patches=M, overlap_frac=0.2,
            seed=42 * 1000 + trial * 100 + M
        )
        p_s, _, _, _ = sequential_quilt(pats, X_f, r=3, K=3)
        p_h, _, _ = hierarchical_quilt(pats, X_f, r=3, K=3)
        seq_vals.append(adjusted_rand_score(lbls, p_s))
        hier_vals.append(adjusted_rand_score(lbls, p_h))
    aris_seq.append((np.mean(seq_vals), np.std(seq_vals) / np.sqrt(n_trials_sweep)))
    aris_hier.append((np.mean(hier_vals), np.std(hier_vals) / np.sqrt(n_trials_sweep)))

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.array(patch_counts)
seq_m = np.array([a[0] for a in aris_seq])
seq_e = np.array([a[1] for a in aris_seq])
hier_m = np.array([a[0] for a in aris_hier])
hier_e = np.array([a[1] for a in aris_hier])

ax.errorbar(x, seq_m, yerr=seq_e, fmt='o-', color='#d62728', lw=2, capsize=4,
            label='Sequential', markersize=6)
ax.errorbar(x, hier_m, yerr=hier_e, fmt='s-', color='#1f77b4', lw=2, capsize=4,
            label='Hierarchical', markersize=6)
ax.fill_between(x, seq_m - seq_e, seq_m + seq_e, color='#d62728', alpha=0.1)
ax.fill_between(x, hier_m - hier_e, hier_m + hier_e, color='#1f77b4', alpha=0.1)

ax.set_xlabel('Number of patches')
ax.set_ylabel('ARI')
ax.set_title('Sequential vs. Hierarchical: ARI vs. Patch Count (20% overlap, 10 replicates)',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(patch_counts)
plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Potential Solutions

The fundamental issue is that greedy pairing optimizes *local* merge quality at the expense of *global* connectivity. Below we outline several approaches to fix this, ordered from most targeted to most general.

---

### 9.1 Overlap-Aware Tree Construction

**Problem**: Greedy max-overlap pairing produces a tree where the root merge is starved of shared samples.

**Solution**: Instead of greedy pairing, construct the merge tree to *guarantee* minimum overlap at every level. This can be formulated as a constrained optimization:

$$\max_{\text{tree } T} \min_{\text{merge } (a,b) \in T} \frac{|\mathcal{S}_a \cap \mathcal{S}_b|}{\max(|\mathcal{S}_a|, |\mathcal{S}_b|)}$$

A practical heuristic: **balanced partitioning** — instead of pairing adjacent patches, interleave them:

```
Patches:  P1 P2 P3 P4 P5 P6 P7 P8
Pair:     (P1,P5) (P2,P6) (P3,P7) (P4,P8)  ← far-apart pairs
```

This ensures each merge bridges a wide sample range, but each pair has *less* overlap at level 0 (the tradeoff).

---

### 9.2 Procrustes Alignment (Orthogonally Constrained $G$)

**Problem**: The unconstrained least-squares $G$ can distort the embedding (shear, scale) when the overlap is small or ill-conditioned.

**Solution**: Replace `lstsq` with **orthogonal Procrustes**, which constrains $G \in O(r)$:

$$\min_{Q \in O(r)} \| U_b[\mathcal{O},:] Q - U_a[\mathcal{O},:] \|_F$$

Solved in closed form via SVD of the cross-covariance matrix:

$$U_a[\mathcal{O},:]^\top U_b[\mathcal{O},:] = W \Sigma Z^\top \quad \Longrightarrow \quad Q^* = Z W^\top$$

**Advantages**: orthogonal $Q$ preserves distances and angles, preventing the scale inflation that ill-conditioned least-squares produces. The solution always exists and is unique (up to reflections).

**Tradeoff**: less flexible than general $G$ — cannot absorb scale differences between patches. But for cluster quilting, scale preservation is usually desirable.

In [ ]:
# === 9.2 Demo: Procrustes vs. least-squares under poor overlap ===

def compare_alignment_methods(N=300, p=20, r=2, K=3, overlap_size=6, seed=7):
    """Compare lstsq vs. Procrustes alignment with small overlap."""
    rng_l = np.random.default_rng(seed)
    
    # Generate clustered data
    labels_l = np.repeat(np.arange(K), N // K)
    angles = np.linspace(0, 2 * np.pi, K, endpoint=False)
    centers = np.column_stack([np.cos(angles), np.sin(angles)]) * 4
    U_true_l = centers[labels_l] + rng_l.standard_normal((N, r)) * 0.4
    
    # Two patches with small overlap
    mid = N // 2
    idx_a_l = np.arange(0, mid + overlap_size)
    idx_b_l = np.arange(mid - overlap_size, N)
    
    V_a_l = rng_l.standard_normal((p, r))
    V_b_l = rng_l.standard_normal((p, r))
    X_a_l = U_true_l[idx_a_l] @ V_a_l.T + rng_l.standard_normal((len(idx_a_l), p)) * 0.1
    X_b_l = U_true_l[idx_b_l] @ V_b_l.T + rng_l.standard_normal((len(idx_b_l), p)) * 0.1
    
    Ua_l, _, _ = np.linalg.svd(X_a_l, full_matrices=False)
    Ub_l, _, _ = np.linalg.svd(X_b_l, full_matrices=False)
    Ua_r_l = Ua_l[:, :r]
    Ub_r_l = Ub_l[:, :r]
    
    # Overlap indices
    ov_a = np.arange(mid, mid + overlap_size)  # local in A
    ov_b = np.arange(0, overlap_size)           # local in B
    
    # Least squares
    G_ls, _, _, _ = np.linalg.lstsq(Ub_r_l[ov_b], Ua_r_l[ov_a], rcond=None)
    Ub_ls = Ub_r_l @ G_ls
    
    # Procrustes
    Q_pro, _ = orthogonal_procrustes(Ub_r_l[ov_b], Ua_r_l[ov_a])
    Ub_pro = Ub_r_l @ Q_pro
    
    return Ua_r_l, Ub_r_l, Ub_ls, Ub_pro, idx_a_l, idx_b_l, labels_l, G_ls, Q_pro


Ua_r, Ub_r, Ub_ls, Ub_pro, idx_a, idx_b, labels_c, G_ls, Q_pro = \
    compare_alignment_methods(overlap_size=5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors_cluster = ['#e41a1c', '#377eb8', '#4daf4a']

titles = ['Unaligned', 'Least-squares $G$', 'Procrustes $Q$']
Ub_versions = [Ub_r, Ub_ls, Ub_pro]

for ax, Ub_v, title in zip(axes, Ub_versions, titles):
    for k in range(3):
        ma = labels_c[idx_a] == k
        mb = labels_c[idx_b] == k
        ax.scatter(Ua_r[ma, 0], Ua_r[ma, 1], c=colors_cluster[k], s=8, alpha=0.4, marker='o')
        ax.scatter(Ub_v[mb, 0], Ub_v[mb, 1], c=colors_cluster[k], s=8, alpha=0.4, marker='^')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')

fig.suptitle('Procrustes vs. Least-Squares with Only 5 Overlap Samples', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Show that G_ls can have large singular values (distortion) while Q_pro doesn't
print(f'Least-squares G singular values: {np.linalg.svd(G_ls, compute_uv=False)}')
print(f'Procrustes Q singular values:    {np.linalg.svd(Q_pro, compute_uv=False)}')
print(f'(Procrustes is always [1, 1, ...] — no distortion)')

---

### 9.3 Global Joint Embedding via Iterative Refinement

**Problem**: Both sequential and hierarchical methods perform *one-shot* pairwise alignment. Errors from early merges are never corrected.

**Solution**: After the initial quilting pass, perform **iterative refinement**: cycle through all patches, re-aligning each patch's $U$ against the current global $\widetilde{U}$, repeating until convergence.

This is analogous to coordinate descent or the Iterative Closest Point (ICP) algorithm in point cloud registration. Each iteration:

1. For each patch $m$: re-solve $G_m$ using the current $\widetilde{U}$ on the overlap
2. Update $\widetilde{U}[\text{new}_m,:] = U_r^{(m)}[\text{new}_m,:] \, G_m$
3. Repeat until $\|\widetilde{U}^{(t+1)} - \widetilde{U}^{(t)}\|_F < \epsilon$

**Advantages**: corrects errors from the initial alignment; converges to a globally consistent embedding.

**Risks**: may not converge for poorly overlapping patches; adds $O(T \cdot M)$ cost for $T$ iterations.

---

### 9.4 Spectral Alignment (Eigenvector Stitching)

**Problem**: Pairwise alignment is inherently local — it only uses information from two patches at a time.

**Solution**: Construct a **sample similarity graph** where edge weights are derived from within-patch SVD similarity, then compute the global embedding via spectral methods (Laplacian eigenvectors).

For each pair of samples $(i, j)$ that co-occur in some patch $m$:

$$w_{ij} = \exp\left(-\frac{\|U_r^{(m)}[i,:] - U_r^{(m)}[j,:]\|^2}{2\sigma^2}\right)$$

This produces a sparse similarity matrix that encodes local geometry from each patch. The top eigenvectors of the normalized Laplacian give a global embedding that respects all patches simultaneously, without any sequential or hierarchical alignment.

**Advantages**: inherently global; no alignment step; no sensitivity to merge order or tree structure.

**Tradeoffs**: requires choosing bandwidth $\sigma$; Laplacian eigendecomposition is $O(N^2)$ for dense graphs; loses the elegant algebraic structure of the SVD-based approach.

---

### 9.5 Regularized Alignment ($G$ with Tikhonov Penalty)

**Problem**: when the overlap is ill-conditioned, $G$ has large singular values that distort the embedding.

**Solution**: add a Tikhonov (ridge) penalty that shrinks $G$ toward orthogonality:

$$\min_G \| U_b[\mathcal{O},:] G - U_a[\mathcal{O},:] \|_F^2 + \lambda \| G - I \|_F^2$$

The closed-form solution is:

$$G^* = (U_b^\top U_b + \lambda I)^{-1} (U_b^\top U_a + \lambda I)$$

For $\lambda \to 0$, this recovers ordinary least-squares. For $\lambda \to \infty$, $G \to I$ (no alignment). Intermediate $\lambda$ values prevent the catastrophic distortion of ill-conditioned least-squares while allowing rotation/reflection.

**Choosing $\lambda$**: can be set proportional to the inverse condition number of $U_b[\mathcal{O},:]$, providing adaptive regularization that kicks in only when needed.

In [ ]:
# === 9.5 Demo: Tikhonov-regularized alignment ===

def tikhonov_align(Ub_ov, Ua_ov, lam):
    """Solve min ||Ub_ov @ G - Ua_ov||^2 + lam * ||G - I||^2."""
    r = Ub_ov.shape[1]
    A = Ub_ov.T @ Ub_ov + lam * np.eye(r)
    B = Ub_ov.T @ Ua_ov + lam * np.eye(r)
    return np.linalg.solve(A, B)


# Reuse the comparison data from 9.2 with 5 overlap samples
Ua_r2, Ub_r2, _, _, idx_a2, idx_b2, labels_c2, _, _ = \
    compare_alignment_methods(overlap_size=5)

mid2 = 150
ov_a2 = np.arange(mid2, mid2 + 5)
ov_b2 = np.arange(0, 5)

lambdas = [0, 0.01, 0.1, 1.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(15, 3.8))

for ax, lam in zip(axes, lambdas):
    if lam == 0:
        G_reg, _, _, _ = np.linalg.lstsq(Ub_r2[ov_b2], Ua_r2[ov_a2], rcond=None)
    else:
        G_reg = tikhonov_align(Ub_r2[ov_b2], Ua_r2[ov_a2], lam)
    
    Ub_reg = Ub_r2 @ G_reg
    svs = np.linalg.svd(G_reg, compute_uv=False)
    
    for k in range(3):
        ma = labels_c2[idx_a2] == k
        mb = labels_c2[idx_b2] == k
        ax.scatter(Ua_r2[ma, 0], Ua_r2[ma, 1], c=colors_cluster[k], s=8, alpha=0.4, marker='o')
        ax.scatter(Ub_reg[mb, 0], Ub_reg[mb, 1], c=colors_cluster[k], s=8, alpha=0.4, marker='^')
    ax.set_title(f'$\\lambda={lam}$\n$\\sigma(G)$=[{svs[0]:.2f}, {svs[1]:.2f}]')
    ax.set_aspect('equal')
    ax.set_xlabel('Component 1')

axes[0].set_ylabel('Component 2')
fig.suptitle('Tikhonov Regularization: Controlling Alignment Distortion', fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---

### 9.6 Summary of Proposed Solutions

| Solution | Addresses | Complexity | Key Tradeoff |
|----------|-----------|------------|--------------|
| **Overlap-aware tree** | Global separation | $O(M^2)$ tree construction | Less overlap at leaf merges |
| **Procrustes** | Ill-conditioned $G$ | Same as lstsq | No scale adaptation |
| **Iterative refinement** | Error accumulation | $O(T \cdot M \cdot r^2)$ | Convergence not guaranteed |
| **Spectral (Laplacian)** | All alignment issues | $O(N^2)$ | Loses SVD structure |
| **Tikhonov regularization** | Ill-conditioned $G$ | Same as lstsq | Requires tuning $\lambda$ |

The most promising **practical** combination: **Procrustes alignment** (drop-in replacement for lstsq, zero tuning) combined with **overlap-aware tree construction** (fixes the root cause of hierarchical failure). Iterative refinement can then be applied as a post-processing step to further reduce residual errors.

<a id='10'></a>
## 10. Summary

### The Sequential Failure Mode (Known)

Error compounds along an $O(M)$-step chain. Each alignment $G_m$ introduces noise $\Delta G_m$, and the effective transformation for late-chain patches is the product $\prod_{j=2}^{k} (G_j^* + \Delta G_j)$. Total error grows linearly with $M$.

### The Hierarchical Promise

Replace the $O(M)$ chain with an $O(\log_2 M)$ tree. Under the assumption of uniform per-merge error, this reduces total error by a factor of $M / \log_2 M$.

### Why Hierarchical Fails

The assumption of uniform per-merge error is violated by greedy max-overlap pairing:

1. **Greedy pairing is locally optimal**: adjacent patches have the most overlap, so they merge first.
2. **This creates geographic branches**: the tree's left subtree covers samples $[0, N/2)$ and the right covers $[N/2, N)$.
3. **The root merge is starved**: the two halves share only boundary samples. The overlap-to-node-size ratio is $\sim f / (M/2)$, potentially below the critical threshold.
4. **A single bad root merge corrupts half the embedding**: unlike the chain (where later patches are most affected), the tree's failure is *catastrophic* — it misaligns an entire branch.

### The Fundamental Tension

$$\underbrace{\text{Fewer alignment steps}}_{O(\log M) \text{ tree depth}} \quad \text{vs.} \quad \underbrace{\text{Worse per-step quality}}_{\text{overlap attenuation at higher levels}}$$

The tree's depth advantage is real, but the greedy pairing strategy ensures that the **critical root merge — which affects the most samples — has the worst alignment quality**. The sequential chain distributes error more evenly: every step has comparable overlap quality, and no single step affects more than one patch's worth of new samples.

### Takeaway

Hierarchical quilting fails not because the tree structure is inherently worse, but because **greedy pairing produces a pathological tree**. The fix is not to abandon hierarchy, but to construct better trees (overlap-aware balancing) and use more robust alignment methods (Procrustes, regularization, iterative refinement).